# Impacts of Borderline Cleaning

### Loading the 24H data

In [1]:
import pickle
import numpy as np

def load_split(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data["X"].astype(np.float32), data["y"]

# Load Hybrid 6h only
folder = "./final_split_data_HybridNorm"

X_train, y_train = load_split(f"{folder}/train_set.pkl")
X_val,   y_val   = load_split(f"{folder}/val_set.pkl")
X_test,  y_test  = load_split(f"{folder}/test_set.pkl")

datasets = {
    "hybrid": {
        "X_train": X_train, "y_train": y_train,
        "X_val": X_val,     "y_val": y_val,
        "X_test": X_test,   "y_test": y_test,
    }
}

# Summary
print(f"{'Dataset':<12} {'Train shape':<25} {'Val shape':<22} {'Test shape':<22} {'Class 0':>8} {'Class 1':>8} {'Ratio':>8}")
print("─" * 110)

d = datasets["hybrid"]
counts = np.bincount(d["y_train"].astype(int), minlength=2)
ratio = counts[0] / counts[1]

print(
    f"{'hybrid':<12} "
    f"{str(d['X_train'].shape):<25} "
    f"{str(d['X_val'].shape):<22} "
    f"{str(d['X_test'].shape):<22} "
    f"{counts[0]:>8} "
    f"{counts[1]:>8} "
    f"{ratio:>7.1f}x"
)

Dataset      Train shape               Val shape              Test shape              Class 0  Class 1    Ratio
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
hybrid       (12455, 288, 10)          (1780, 288, 10)        (3559, 288, 10)           12337      118   104.6x


### TomekLinks - EditedNearestNeighbours - NearMiss 

In [2]:
import numpy as np
from imblearn.under_sampling import TomekLinks, EditedNearestNeighbours, NearMiss

# ══════════════════════════════════════════════════════════════
# Apply TomekLinks, ENN, and NearMiss to loaded Hybrid 6h data
# Final datasets remain in 3D shape: (N, T, F)
# ══════════════════════════════════════════════════════════════

d = datasets["hybrid"]

X_train = d["X_train"]
y_train = d["y_train"]

X_val = d["X_val"]
y_val = d["y_val"]

X_test = d["X_test"]
y_test = d["y_test"]

N, T, F = X_train.shape

# Flatten only for imblearn samplers
X_train_flat = X_train.reshape(N, T * F)

original_counts = np.bincount(y_train.astype(int), minlength=2)
original_total = len(y_train)

samplers = {
    "tomek": TomekLinks(),
    "enn": EditedNearestNeighbours(n_neighbors=3),
    "nearmiss": NearMiss(version=3),
}

cleaned_datasets = {}

print("Original training set:")
print("Train shape:", X_train.shape)
print("Class 0:", original_counts[0])
print("Class 1:", original_counts[1])
print(f"Ratio: {original_counts[0] / original_counts[1]:.1f}x")
print("─" * 70)

for name, sampler in samplers.items():

    print(f"\nApplying {name}...")

    X_train_res_flat, y_train_res = sampler.fit_resample(X_train_flat, y_train)

    # Reshape back to time-series format
    X_train_res = X_train_res_flat.reshape(-1, T, F).astype(np.float32)

    cleaned_datasets[name] = {
        "X_train": X_train_res,
        "y_train": y_train_res,

        "X_val": X_val,
        "y_val": y_val,

        "X_test": X_test,
        "y_test": y_test,
    }

    new_counts = np.bincount(y_train_res.astype(int), minlength=2)
    new_total = len(y_train_res)

    removed_total = original_total - new_total
    removed_class0 = original_counts[0] - new_counts[0]
    removed_class1 = original_counts[1] - new_counts[1]

    ratio = new_counts[0] / new_counts[1] if new_counts[1] > 0 else np.inf

    print("Before:")
    print("  Train shape:", X_train.shape)
    print("  Class 0:", original_counts[0])
    print("  Class 1:", original_counts[1])

    print("After:")
    print("  Train shape:", X_train_res.shape)
    print("  Class 0:", new_counts[0])
    print("  Class 1:", new_counts[1])
    print(f"  Ratio: {ratio:.1f}x")

    print("Removed:")
    print("  Total removed:", removed_total)
    print("  Class 0 removed:", removed_class0)
    print("  Class 1 removed:", removed_class1)

print("\n✅ Three cleaned time-series datasets are ready:")
print("cleaned_datasets['tomek']")
print("cleaned_datasets['enn']")
print("cleaned_datasets['nearmiss']")

Original training set:
Train shape: (12455, 288, 10)
Class 0: 12337
Class 1: 118
Ratio: 104.6x
──────────────────────────────────────────────────────────────────────

Applying tomek...
Before:
  Train shape: (12455, 288, 10)
  Class 0: 12337
  Class 1: 118
After:
  Train shape: (12428, 288, 10)
  Class 0: 12310
  Class 1: 118
  Ratio: 104.3x
Removed:
  Total removed: 27
  Class 0 removed: 27
  Class 1 removed: 0

Applying enn...
Before:
  Train shape: (12455, 288, 10)
  Class 0: 12337
  Class 1: 118
After:
  Train shape: (12268, 288, 10)
  Class 0: 12150
  Class 1: 118
  Ratio: 103.0x
Removed:
  Total removed: 187
  Class 0 removed: 187
  Class 1 removed: 0

Applying nearmiss...
Before:
  Train shape: (12455, 288, 10)
  Class 0: 12337
  Class 1: 118
After:
  Train shape: (236, 288, 10)
  Class 0: 118
  Class 1: 118
  Ratio: 1.0x
Removed:
  Total removed: 12219
  Class 0 removed: 12219
  Class 1 removed: 0

✅ Three cleaned time-series datasets are ready:
cleaned_datasets['tomek']
cleane

### Catch22 Vectorization for SVM

In [3]:
import numpy as np
from joblib import Parallel, delayed
from sktime.transformations.panel.catch22 import Catch22
from sklearn.impute import SimpleImputer
import warnings

# ══════════════════════════════════════════════════════════════
# CATCH22 FEATURE EXTRACTION FOR CLEANED 6H DATASETS
# Input : cleaned_datasets["tomek"], ["enn"], ["nearmiss"]
# Output: datasets_svm_cleaned
# ══════════════════════════════════════════════════════════════

def _catch22_chunk(X_chunk):
    transformer = Catch22(catch24=False)
    return transformer.fit_transform(X_chunk).to_numpy()


def extract_catch22(X, label="", n_jobs=-1, chunk_size=200):

    n_samples, n_timepoints, n_channels = X.shape

    # Replace NaN / Inf before Catch22
    if not np.isfinite(X).all():
        n_bad = (~np.isfinite(X)).sum()
        print(f"    ⚠️ {n_bad} non-finite values in {label} — replacing with channel median")

        X = X.copy()

        for c in range(n_channels):
            col = X[:, :, c]
            median_c = np.nanmedian(col)

            if not np.isfinite(median_c):
                median_c = 0.0

            col[~np.isfinite(col)] = median_c
            X[:, :, c] = col

    # sktime expects shape: (samples, channels, timesteps)
    X_sktime = X.transpose(0, 2, 1).astype(np.float64)

    chunks = [
        X_sktime[i:i + chunk_size]
        for i in range(0, n_samples, chunk_size)
    ]

    print(f"    {label}: {n_samples} samples in {len(chunks)} chunks", flush=True)

    results = Parallel(n_jobs=n_jobs)(
        delayed(_catch22_chunk)(chunk)
        for chunk in chunks
    )

    X_feat = np.vstack(results)

    expected_cols = 22 * n_channels

    assert X_feat.shape == (n_samples, expected_cols), (
        f"Unexpected shape for {label}: {X_feat.shape}, "
        f"expected ({n_samples}, {expected_cols})"
    )

    return X_feat.astype(np.float32)


# ══════════════════════════════════════════════════════════════
# Sanity check before extraction
# ══════════════════════════════════════════════════════════════

print("Pre-extraction value range check:")
print(f"{'Dataset':<12} {'Train shape':<22} {'Finite?':<10} {'Min':>12} {'Max':>12} {'NaN/Inf':>10}")
print("─" * 85)

for dataset_name, d in cleaned_datasets.items():
    X = d["X_train"]

    print(
        f"{dataset_name:<12} "
        f"{str(X.shape):<22} "
        f"{str(np.isfinite(X).all()):<10} "
        f"{np.nanmin(X):>12.4f} "
        f"{np.nanmax(X):>12.4f} "
        f"{(~np.isfinite(X)).sum():>10}"
    )


# ══════════════════════════════════════════════════════════════
# Extract Catch22 features
# ══════════════════════════════════════════════════════════════

datasets_svm_cleaned = {}

print("\nExtracting Catch22 features for cleaned datasets...")

for dataset_name, d in cleaned_datasets.items():

    print(f"\n[{dataset_name}] extracting train...")
    X_tr = extract_catch22(
        d["X_train"],
        label=f"{dataset_name}/train",
        n_jobs=-1,
        chunk_size=200
    )

    print(f"[{dataset_name}] extracting val...")
    X_va = extract_catch22(
        d["X_val"],
        label=f"{dataset_name}/val",
        n_jobs=-1,
        chunk_size=200
    )

    print(f"[{dataset_name}] extracting test...")
    X_te = extract_catch22(
        d["X_test"],
        label=f"{dataset_name}/test",
        n_jobs=-1,
        chunk_size=200
    )

    datasets_svm_cleaned[dataset_name] = {
        "X_train": X_tr,
        "y_train": d["y_train"],

        "X_val": X_va,
        "y_val": d["y_val"],

        "X_test": X_te,
        "y_test": d["y_test"],
    }

    print(
        f"[{dataset_name}] ✓ "
        f"train {X_tr.shape} | "
        f"val {X_va.shape} | "
        f"test {X_te.shape}"
    )


# ══════════════════════════════════════════════════════════════
# Impute Catch22 features
# Fit imputer on train only, apply to val/test
# ══════════════════════════════════════════════════════════════

print("\nChecking and imputing Catch22 features...")

for dataset_name in list(datasets_svm_cleaned.keys()):

    d = datasets_svm_cleaned[dataset_name]

    X_tr = d["X_train"].copy()
    X_va = d["X_val"].copy()
    X_te = d["X_test"].copy()

    y_tr = d["y_train"]
    y_va = d["y_val"]
    y_te = d["y_test"]

    n_bad_tr = (~np.isfinite(X_tr)).sum()
    n_bad_va = (~np.isfinite(X_va)).sum()
    n_bad_te = (~np.isfinite(X_te)).sum()

    if n_bad_tr > 0 or n_bad_va > 0 or n_bad_te > 0:

        print(
            f"  ⚠️ [{dataset_name}] non-finite values — "
            f"train: {n_bad_tr}, val: {n_bad_va}, test: {n_bad_te} → imputing"
        )

        X_tr = np.where(np.isfinite(X_tr), X_tr, np.nan)
        X_va = np.where(np.isfinite(X_va), X_va, np.nan)
        X_te = np.where(np.isfinite(X_te), X_te, np.nan)

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")

            imputer = SimpleImputer(strategy="median")

            X_tr = imputer.fit_transform(X_tr)
            X_va = imputer.transform(X_va)
            X_te = imputer.transform(X_te)

        X_tr = np.nan_to_num(X_tr, nan=0.0, posinf=0.0, neginf=0.0)
        X_va = np.nan_to_num(X_va, nan=0.0, posinf=0.0, neginf=0.0)
        X_te = np.nan_to_num(X_te, nan=0.0, posinf=0.0, neginf=0.0)

        print(f"  ✓ [{dataset_name}] imputed")

    else:
        print(f"  ✓ [{dataset_name}] no NaN or Inf")

    datasets_svm_cleaned[dataset_name] = {
        "X_train": X_tr.astype(np.float32),
        "y_train": y_tr,

        "X_val": X_va.astype(np.float32),
        "y_val": y_va,

        "X_test": X_te.astype(np.float32),
        "y_test": y_te,
    }


print("\n✅ SVM-ready Catch22 datasets are ready:")
print(list(datasets_svm_cleaned.keys()))

for name, d in datasets_svm_cleaned.items():
    counts = np.bincount(d["y_train"].astype(int), minlength=2)
    ratio = counts[0] / counts[1] if counts[1] > 0 else np.inf

    print(
        f"{name:<10} "
        f"train {d['X_train'].shape} | "
        f"val {d['X_val'].shape} | "
        f"test {d['X_test'].shape} | "
        f"class0={counts[0]} class1={counts[1]} ratio={ratio:.1f}x"
    )

Pre-extraction value range check:
Dataset      Train shape            Finite?             Min          Max    NaN/Inf
─────────────────────────────────────────────────────────────────────────────────────
tomek        (12428, 288, 10)       True            -6.5136      27.8326          0
enn          (12268, 288, 10)       True            -6.5136      27.8326          0
nearmiss     (236, 288, 10)         True            -6.5136      27.2851          0

Extracting Catch22 features for cleaned datasets...

[tomek] extracting train...
    tomek/train: 12428 samples in 63 chunks
[tomek] extracting val...
    tomek/val: 1780 samples in 9 chunks
[tomek] extracting test...
    tomek/test: 3559 samples in 18 chunks
[tomek] ✓ train (12428, 220) | val (1780, 220) | test (3559, 220)

[enn] extracting train...
    enn/train: 12268 samples in 62 chunks
[enn] extracting val...
    enn/val: 1780 samples in 9 chunks
[enn] extracting test...
    enn/test: 3559 samples in 18 chunks
[enn] ✓ train (12268,

## Classification

### Helper Functions

In [4]:
# ══════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ══════════════════════════════════════════════════════════════
import numpy as np
from sklearn.metrics import confusion_matrix
from sklearn.svm import SVC
import time
import os

RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)


def compute_metrics(y_true, y_pred):
    cm             = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm.ravel()

    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    accuracy  = (TP + TN) / (TP + TN + FP + FN)

    tss  = recall - FP / (FP + TN) if (FP + TN) > 0 else 0.0

    expected = ((TP + FN) * (TP + FP) + (TN + FP) * (TN + FN)) / (TP + TN + FP + FN) ** 2
    hss1     = (accuracy - expected) / (1 - expected) if (1 - expected) > 0 else 0.0

    denom = ((TP + FN) * (FN + TN) + (TP + FP) * (FP + TN))
    hss2  = 2 * (TP * TN - FP * FN) / denom if denom > 0 else 0.0

    hits_random = (TP + FP) * (TP + FN) / (TP + TN + FP + FN)
    gss = (TP - hits_random) / (TP + FP + FN - hits_random) if (TP + FP + FN - hits_random) > 0 else 0.0

    return {
        'TP': int(TP), 'TN': int(TN), 'FP': int(FP), 'FN': int(FN),
        'tss': tss, 'hss1': hss1, 'hss2': hss2, 'gss': gss,
        'recall': recall, 'f1': f1, 'accuracy': accuracy
    }


def train_and_evaluate(model, X_train, y_train, X_test, y_test):
    t0         = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - t0

    t0         = time.time()
    y_pred     = model.predict(X_test)
    infer_time = time.time() - t0

    metrics = compute_metrics(y_test, y_pred)
    metrics['train_time'] = train_time
    metrics['infer_time'] = infer_time
    return metrics


def save_results(metrics_list, filepath):
    with open(filepath, 'w') as f:
        for m in metrics_list:
            line = (f"{m['TP']},{m['TN']},{m['FP']},{m['FN']},"
                    f"{m['tss']:.6f},{m['hss1']:.6f},{m['hss2']:.6f},{m['gss']:.6f},"
                    f"{m['recall']:.6f},{m['f1']:.6f},{m['accuracy']:.6f},"
                    f"{m['train_time']:.4f},{m['infer_time']:.4f}")
            f.write(line + "\n")


def print_results(metrics_list, title):
    keys = ['tss', 'hss1', 'hss2', 'gss', 'recall', 'f1', 'accuracy', 'train_time', 'infer_time']
    print(f"\n{'─'*55}")
    print(f"  {title}")
    print(f"{'─'*55}")
    for i, m in enumerate(metrics_list):
        print(f"  Run {i+1}: TP={m['TP']}  TN={m['TN']}  FP={m['FP']}  FN={m['FN']}")
        print(f"         TSS={m['tss']:.4f}  HSS1={m['hss1']:.4f}  HSS2={m['hss2']:.4f}  GSS={m['gss']:.4f}")
        print(f"         Recall={m['recall']:.4f}  F1={m['f1']:.4f}  Acc={m['accuracy']:.4f}")
        print(f"         Train={m['train_time']:.2f}s  Infer={m['infer_time']:.4f}s")
        print()
    print(f"  ── Average of {len(metrics_list)} runs ──")
    for k in keys:
        avg = np.mean([m[k] for m in metrics_list])
        print(f"  {k:<12} : {avg:.4f}")
    print(f"{'─'*55}")

### SVM

In [5]:
import os
import copy
import warnings
import numpy as np

from sklearn.svm import SVC
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# ══════════════════════════════════════════════════════════════
# SVM FINAL EXPERIMENTS — CLEANED HYBRID 6H DATA
# Uses fixed best hyperparameters selected previously
# Best model: RBF-SVC, C=0.1, gamma=scale, class_weight=balanced
# Datasets: tomek, enn, nearmiss
# ══════════════════════════════════════════════════════════════

RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ---------------------------------------------------------
# Fixed best SVM hyperparameters
# ---------------------------------------------------------
best_model_class = SVC

best_params = {
    "kernel": "rbf",
    "C": 0.1,
    "gamma": "scale",
    "class_weight": "balanced",
    "cache_size": 2000
}

print("═" * 70)
print("Final SVM experiments on cleaned Hybrid datasets")
print("Using fixed best hyperparameters")
print("═" * 70)
print(f"Model  : {best_model_class.__name__}")
print(f"Params : {best_params}")


# ---------------------------------------------------------
# Run final experiments on cleaned datasets
# ---------------------------------------------------------
all_cleaned_results = {}

for dataset_key, d in datasets_svm_cleaned.items():

    print("\n" + "─" * 70)
    print(f"Dataset : {dataset_key}")
    print(f"Model   : {best_model_class.__name__}")
    print(f"Params  : {best_params}")

    train_counts = np.bincount(d["y_train"].astype(int), minlength=2)
    test_counts  = np.bincount(d["y_test"].astype(int), minlength=2)

    print(f"Train shape : {d['X_train'].shape}")
    print(f"Test shape  : {d['X_test'].shape}")
    print(f"Train class : Non-SEP={train_counts[0]} | SEP={train_counts[1]}")
    print(f"Test class  : Non-SEP={test_counts[0]} | SEP={test_counts[1]}")
    print("─" * 70)

    metrics_list = []

    for run in range(2):

        model = best_model_class(**copy.deepcopy(best_params))

        metrics = train_and_evaluate(
            model,
            d["X_train"], d["y_train"],
            d["X_test"],  d["y_test"]
        )

        metrics_list.append(metrics)

        precision_text = (
            f" | Precision={metrics['precision']:.4f}"
            if "precision" in metrics else ""
        )

        print(
            f"Run {run + 1}: "
            f"TSS={metrics['tss']:.4f} | "
            f"F1={metrics['f1']:.4f} | "
            f"Recall={metrics['recall']:.4f}"
            f"{precision_text} | "
            f"Train={metrics['train_time']:.2f}s"
        )

    all_cleaned_results[dataset_key] = metrics_list

    filepath = os.path.join(RESULTS_DIR, f"svm_hybrid_{dataset_key}.txt")
    save_results(metrics_list, filepath)
    print_results(metrics_list, f"SVM | Hybrid | {dataset_key}")


# ---------------------------------------------------------
# Select best cleaned dataset by average TSS
# ---------------------------------------------------------
best_dataset = None
best_avg_tss = -np.inf

print("\n" + "═" * 70)
print("SVM cleaned Hybrid summary")
print("═" * 70)

print(
    f"{'Dataset':<12} "
    f"{'Avg TSS':>10} "
    f"{'Avg F1':>10} "
    f"{'Avg Recall':>12}"
)
print("─" * 50)

for dataset_key, metrics_list in all_cleaned_results.items():

    avg_tss = np.mean([m["tss"] for m in metrics_list])
    avg_f1 = np.mean([m["f1"] for m in metrics_list])
    avg_recall = np.mean([m["recall"] for m in metrics_list])

    print(
        f"{dataset_key:<12} "
        f"{avg_tss:>10.4f} "
        f"{avg_f1:>10.4f} "
        f"{avg_recall:>12.4f}"
    )

    if avg_tss > best_avg_tss:
        best_avg_tss = avg_tss
        best_dataset = dataset_key


print("\nBest SVM cleaned Hybrid dataset:")
print(f"  Dataset : {best_dataset}")
print(f"  Avg TSS : {best_avg_tss:.4f}")
print(f"  Model   : {best_model_class.__name__}")
print(f"  Params  : {best_params}")

print("\nAll SVM cleaned Hybrid experiments complete.")
print(f"Results saved to: {os.path.abspath(RESULTS_DIR)}/")

══════════════════════════════════════════════════════════════════════
Final SVM experiments on cleaned Hybrid datasets
Using fixed best hyperparameters
══════════════════════════════════════════════════════════════════════
Model  : SVC
Params : {'kernel': 'rbf', 'C': 0.1, 'gamma': 'scale', 'class_weight': 'balanced', 'cache_size': 2000}

──────────────────────────────────────────────────────────────────────
Dataset : tomek
Model   : SVC
Params  : {'kernel': 'rbf', 'C': 0.1, 'gamma': 'scale', 'class_weight': 'balanced', 'cache_size': 2000}
Train shape : (12428, 220)
Test shape  : (3559, 220)
Train class : Non-SEP=12310 | SEP=118
Test class  : Non-SEP=3525 | SEP=34
──────────────────────────────────────────────────────────────────────
Run 1: TSS=0.4812 | F1=0.0784 | Recall=0.6176 | Train=8.55s
Run 2: TSS=0.4812 | F1=0.0784 | Recall=0.6176 | Train=8.33s

───────────────────────────────────────────────────────
  SVM | Hybrid | tomek
───────────────────────────────────────────────────────


### GRU 

In [6]:
# ══════════════════════════════════════════════════════════════
# HELPER FUNCTIONS — PyTorch
# ══════════════════════════════════════════════════════════════
import torch
import torch.nn as nn
import numpy as np
import time
import os
import warnings
warnings.filterwarnings('ignore')

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")


class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1, dropout=0.3):
        super(GRUModel, self).__init__()
        self.gru      = nn.GRU(input_size, hidden_size, num_layers=num_layers,
                               batch_first=True,
                               dropout=dropout if num_layers > 1 else 0)
        self.bn       = nn.BatchNorm1d(hidden_size)
        self.dropout  = nn.Dropout(dropout)
        self.fc1      = nn.Linear(hidden_size, 32)
        self.relu     = nn.ReLU()
        self.dropout2 = nn.Dropout(0.2)
        self.fc2      = nn.Linear(32, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        out    = out[:, -1, :]
        out    = self.bn(out)
        out    = self.dropout(out)
        out    = self.relu(self.fc1(out))
        out    = self.dropout2(out)
        return self.fc2(out).squeeze()


def train_and_evaluate_gru(params, X_train, y_train, X_val, y_val, X_test, y_test, class_ratio=13):
    X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_va = torch.tensor(X_val,   dtype=torch.float32).to(device)
    y_va = torch.tensor(y_val,   dtype=torch.float32).to(device)
    X_te = torch.tensor(X_test,  dtype=torch.float32).to(device)

    input_size = X_train.shape[2]
    model      = GRUModel(input_size,
                          hidden_size = params["units"],
                          num_layers  = params["layers"],
                          dropout     = params["dropout"]).to(device)

    pos_weight = torch.tensor([float(class_ratio)], dtype=torch.float32).to(device)
    criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer  = torch.optim.Adam(model.parameters(), lr=params["lr"])

    batch_size     = params["batch_size"]
    n_samples      = X_tr.shape[0]
    best_val_loss  = float('inf')
    patience_count = 0
    best_state     = None

    t0 = time.time()
    for epoch in range(50):
        model.train()
        perm = torch.randperm(n_samples)
        for i in range(0, n_samples, batch_size):
            idx    = perm[i:i + batch_size]
            xb, yb = X_tr[idx], y_tr[idx]
            optimizer.zero_grad()
            loss   = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_va), y_va).item()

        if val_loss < best_val_loss:
            best_val_loss  = val_loss
            patience_count = 0
            best_state     = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_count += 1
            if patience_count >= 5:
                break

    train_time = time.time() - t0

    model.load_state_dict(best_state)
    model.eval()
    t0 = time.time()
    with torch.no_grad():
        y_prob = torch.sigmoid(model(X_te)).cpu().numpy()
    y_pred     = (y_prob >= 0.5).astype(int)
    infer_time = time.time() - t0

    metrics = compute_metrics(y_test.astype(int), y_pred)
    metrics['train_time'] = train_time
    metrics['infer_time'] = infer_time
    return metrics, model


Using device: mps


### Running the GRU Experiments

In [8]:
import os
import numpy as np
import copy

# ══════════════════════════════════════════════════════════════
# GRU FINAL EXPERIMENTS — CLEANED HYBRID 6H DATA
# Uses fixed best hyperparameters selected previously
# Best params: units=128, layers=1, dropout=0.3, lr=0.001, batch_size=32
# Datasets: tomek, enn, nearmiss
# ══════════════════════════════════════════════════════════════

RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ---------------------------------------------------------
# Fixed best GRU hyperparameters
# ---------------------------------------------------------
best_params = {
    "units": 128,
    "layers": 1,
    "dropout": 0.3,
    "lr": 0.001,
    "batch_size": 32
}

print(f"{'═' * 70}")
print("  Classifier : GRU (PyTorch)")
print("  Experiment : Cleaned Hybrid data")
print(f"  Best params: {best_params}")
print(f"{'═' * 70}")


# ---------------------------------------------------------
# Run final GRU experiments on cleaned datasets
# ---------------------------------------------------------
all_gru_cleaned_results = {}

for dataset_key, d in cleaned_datasets.items():

    timesteps = d["X_train"].shape[1]

    train_counts = np.bincount(d["y_train"].astype(int), minlength=2)
    val_counts   = np.bincount(d["y_val"].astype(int), minlength=2)
    test_counts  = np.bincount(d["y_test"].astype(int), minlength=2)

    cr = train_counts[0] / train_counts[1]

    print("\n" + "─" * 70)
    print(f"Dataset   : {dataset_key}")
    print(f"Timesteps : {timesteps}")
    print(f"Train     : {d['X_train'].shape} | Non-SEP={train_counts[0]} | SEP={train_counts[1]}")
    print(f"Val       : {d['X_val'].shape} | Non-SEP={val_counts[0]} | SEP={val_counts[1]}")
    print(f"Test      : {d['X_test'].shape} | Non-SEP={test_counts[0]} | SEP={test_counts[1]}")
    print(f"Ratio     : {cr:.1f}:1")
    print(f"Params    : {best_params}")
    print("─" * 70)

    metrics_list = []

    for run in range(2):

        print(f"Run {run + 1} …", flush=True)

        metrics, _ = train_and_evaluate_gru(
            copy.deepcopy(best_params),
            d["X_train"], d["y_train"],
            d["X_val"],   d["y_val"],
            d["X_test"],  d["y_test"],
            class_ratio=cr
        )

        metrics_list.append(metrics)

        precision_text = (
            f" | Precision={metrics['precision']:.4f}"
            if "precision" in metrics else ""
        )

        print(
            f"Run {run + 1}: "
            f"TSS={metrics['tss']:.4f} | "
            f"F1={metrics['f1']:.4f} | "
            f"Recall={metrics['recall']:.4f}"
            f"{precision_text} | "
            f"Train={metrics['train_time']:.1f}s"
        )

    all_gru_cleaned_results[dataset_key] = metrics_list

    filepath = os.path.join(RESULTS_DIR, f"gru_hybrid_{dataset_key}.txt")
    save_results(metrics_list, filepath)
    print_results(metrics_list, f"GRU | Hybrid | {dataset_key}")


# ---------------------------------------------------------
# Select best cleaned dataset by average TSS
# ---------------------------------------------------------
best_dataset = None
best_avg_tss = -np.inf

print("\n" + "═" * 70)
print("GRU cleaned Hybrid summary")
print("═" * 70)

print(
    f"{'Dataset':<12} "
    f"{'Train N':>10} "
    f"{'Avg TSS':>10} "
    f"{'Avg F1':>10} "
    f"{'Avg Recall':>12}"
)
print("─" * 65)

for dataset_key, metrics_list in all_gru_cleaned_results.items():

    d = cleaned_datasets[dataset_key]

    avg_tss = np.mean([m["tss"] for m in metrics_list])
    avg_f1 = np.mean([m["f1"] for m in metrics_list])
    avg_recall = np.mean([m["recall"] for m in metrics_list])

    print(
        f"{dataset_key:<12} "
        f"{len(d['y_train']):>10} "
        f"{avg_tss:>10.4f} "
        f"{avg_f1:>10.4f} "
        f"{avg_recall:>12.4f}"
    )

    if avg_tss > best_avg_tss:
        best_avg_tss = avg_tss
        best_dataset = dataset_key


print("\nBest GRU cleaned Hybrid dataset:")
print(f"  Dataset : {best_dataset}")
print(f"  Avg TSS : {best_avg_tss:.4f}")
print(f"  Params  : {best_params}")

print("\n✅ All GRU cleaned Hybrid experiments complete.")
print(f"Results saved to: {os.path.abspath(RESULTS_DIR)}/")
print(f"Files: {sorted([f for f in os.listdir(RESULTS_DIR) if f.endswith('.txt')])}")

══════════════════════════════════════════════════════════════════════
  Classifier : GRU (PyTorch)
  Experiment : Cleaned Hybrid data
  Best params: {'units': 128, 'layers': 1, 'dropout': 0.3, 'lr': 0.001, 'batch_size': 32}
══════════════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────────────
Dataset   : tomek
Timesteps : 288
Train     : (12428, 288, 10) | Non-SEP=12310 | SEP=118
Val       : (1780, 288, 10) | Non-SEP=1763 | SEP=17
Test      : (3559, 288, 10) | Non-SEP=3525 | SEP=34
Ratio     : 104.3:1
Params    : {'units': 128, 'layers': 1, 'dropout': 0.3, 'lr': 0.001, 'batch_size': 32}
──────────────────────────────────────────────────────────────────────
Run 1 …
Run 1: TSS=0.6261 | F1=0.1126 | Recall=0.7353 | Train=255.6s
Run 2 …
Run 2: TSS=0.6331 | F1=0.0992 | Recall=0.7647 | Train=226.5s

───────────────────────────────────────────────────────
  GRU | Hybrid | tomek
───────────────────────────────────────────

### PatchTST

In [11]:
import os
import time
import copy
import warnings
import numpy as np

import torch
import torch.nn as nn
from transformers import PatchTSTConfig, PatchTSTForClassification

warnings.filterwarnings("ignore")

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")


def patchtst_label(p):
    return (
        f"patch={p['patch_length']}  "
        f"stride={p['patch_stride']}  "
        f"d={p['d_model']}  "
        f"heads={p['num_attention_heads']}  "
        f"layers={p['num_hidden_layers']}  "
        f"drop={p['dropout']}  "
        f"lr={p['lr']}  "
        f"bs={p['batch_size']}"
    )


def build_patchtst_model(params, seq_len, n_channels):
    config = PatchTSTConfig(
        num_input_channels=n_channels,
        context_length=seq_len,
        num_targets=2,

        patch_length=params["patch_length"],
        patch_stride=params["patch_stride"],

        d_model=params["d_model"],
        num_attention_heads=params["num_attention_heads"],
        num_hidden_layers=params["num_hidden_layers"],
        ffn_dim=params["ffn_dim"],

        attention_dropout=params["dropout"],
        positional_dropout=params["dropout"],
        ff_dropout=params["dropout"],
        head_dropout=params["dropout"],

        pooling_type="mean",
        use_cls_token=True,
        norm_type="layernorm",
        channel_attention=True,
        problem_type="single_label_classification",
        scaling=None,
    )

    return PatchTSTForClassification(config)


def train_and_evaluate_patchtst(
    params,
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
    class_ratio,
    max_epochs=30,
    patience=3
):
    X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train, dtype=torch.long).to(device)

    X_va = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_va = torch.tensor(y_val, dtype=torch.long).to(device)

    X_te = torch.tensor(X_test, dtype=torch.float32).to(device)

    seq_len = X_train.shape[1]
    n_channels = X_train.shape[2]

    model = build_patchtst_model(
        params,
        seq_len=seq_len,
        n_channels=n_channels
    ).to(device)

    class_weights = torch.tensor(
        [1.0, float(class_ratio)],
        dtype=torch.float32
    ).to(device)

    criterion = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=params["lr"],
        weight_decay=1e-4
    )

    batch_size = params["batch_size"]
    n_samples = X_tr.shape[0]

    best_val_loss = float("inf")
    best_state = None
    patience_count = 0

    t0 = time.time()

    for epoch in range(max_epochs):
        model.train()

        perm = torch.randperm(n_samples, device=device)

        for i in range(0, n_samples, batch_size):
            idx = perm[i:i + batch_size]

            xb = X_tr[idx]
            yb = y_tr[idx]

            optimizer.zero_grad()

            outputs = model(past_values=xb)
            logits = outputs.prediction_logits

            loss = criterion(logits, yb)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_outputs = model(past_values=X_va)
            val_logits = val_outputs.prediction_logits
            val_loss = criterion(val_logits, y_va).item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= patience:
                break

    train_time = time.time() - t0

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    t0 = time.time()

    with torch.no_grad():
        test_outputs = model(past_values=X_te)
        test_logits = test_outputs.prediction_logits
        y_pred = torch.argmax(test_logits, dim=1).cpu().numpy()

    infer_time = time.time() - t0

    metrics = compute_metrics(y_test.astype(int), y_pred.astype(int))
    metrics["train_time"] = train_time
    metrics["infer_time"] = infer_time

    return metrics, model

Using device: mps


In [12]:
# ══════════════════════════════════════════════════════════════
# PATCHTST FINAL EXPERIMENTS — CLEANED HYBRID 6H DATA
# Uses fixed best hyperparameters selected previously
# Best params: patch=12, stride=12, d=32, heads=4, layers=1,
#              dropout=0.2, lr=0.001, batch_size=128
# Datasets: tomek, enn, nearmiss
# ══════════════════════════════════════════════════════════════

import os
import copy
import numpy as np

RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ---------------------------------------------------------
# Fixed best PatchTST hyperparameters
# ---------------------------------------------------------
best_params = {
    "patch_length": 12,
    "patch_stride": 12,
    "d_model": 32,
    "num_attention_heads": 4,
    "num_hidden_layers": 1,
    "ffn_dim": 128,
    "dropout": 0.2,
    "lr": 0.001,
    "batch_size": 128,
}


def patchtst_label(p):
    return (
        f"patch={p['patch_length']}  "
        f"stride={p['patch_stride']}  "
        f"d={p['d_model']}  "
        f"heads={p['num_attention_heads']}  "
        f"layers={p['num_hidden_layers']}  "
        f"drop={p['dropout']}  "
        f"lr={p['lr']}  "
        f"bs={p['batch_size']}"
    )


print(f"{'═' * 70}")
print("  Classifier : Hugging Face PatchTST")
print("  Experiment : Cleaned Hybrid data")
print(f"  Best params: {patchtst_label(best_params)}")
print(f"{'═' * 70}")


# ---------------------------------------------------------
# Run final PatchTST experiments on cleaned datasets
# ---------------------------------------------------------
all_patchtst_cleaned_results = {}

for dataset_key, d in cleaned_datasets.items():

    timesteps = d["X_train"].shape[1]

    train_counts = np.bincount(d["y_train"].astype(int), minlength=2)
    val_counts   = np.bincount(d["y_val"].astype(int), minlength=2)
    test_counts  = np.bincount(d["y_test"].astype(int), minlength=2)

    cr = train_counts[0] / train_counts[1]

    print("\n" + "─" * 70)
    print(f"Dataset   : {dataset_key}")
    print(f"Timesteps : {timesteps}")
    print(f"Train     : {d['X_train'].shape} | Non-SEP={train_counts[0]} | SEP={train_counts[1]}")
    print(f"Val       : {d['X_val'].shape} | Non-SEP={val_counts[0]} | SEP={val_counts[1]}")
    print(f"Test      : {d['X_test'].shape} | Non-SEP={test_counts[0]} | SEP={test_counts[1]}")
    print(f"Ratio     : {cr:.1f}:1")
    print(f"Params    : {patchtst_label(best_params)}")
    print("─" * 70)

    metrics_list = []

    for run in range(2):

        print(f"Run {run + 1} …", flush=True)

        metrics, _ = train_and_evaluate_patchtst(
            copy.deepcopy(best_params),
            d["X_train"], d["y_train"],
            d["X_val"],   d["y_val"],
            d["X_test"],  d["y_test"],
            class_ratio=cr,
            max_epochs=30,
            patience=3
        )

        metrics_list.append(metrics)

        precision_text = (
            f" | Precision={metrics['precision']:.4f}"
            if "precision" in metrics else ""
        )

        print(
            f"Run {run + 1}: "
            f"TSS={metrics['tss']:.4f} | "
            f"F1={metrics['f1']:.4f} | "
            f"Recall={metrics['recall']:.4f}"
            f"{precision_text} | "
            f"Train={metrics['train_time']:.1f}s"
        )

    all_patchtst_cleaned_results[dataset_key] = metrics_list

    filepath = os.path.join(RESULTS_DIR, f"patchtst_hybrid_{dataset_key}.txt")
    save_results(metrics_list, filepath)
    print_results(metrics_list, f"PatchTST | Hybrid | {dataset_key}")


# ---------------------------------------------------------
# Select best cleaned dataset by average TSS
# ---------------------------------------------------------
best_dataset = None
best_avg_tss = -np.inf

print("\n" + "═" * 70)
print("PatchTST cleaned Hybrid summary")
print("═" * 70)

print(
    f"{'Dataset':<12} "
    f"{'Train N':>10} "
    f"{'Avg TSS':>10} "
    f"{'Avg F1':>10} "
    f"{'Avg Recall':>12}"
)

print("─" * 65)

for dataset_key, metrics_list in all_patchtst_cleaned_results.items():

    d = cleaned_datasets[dataset_key]

    avg_tss = np.mean([m["tss"] for m in metrics_list])
    avg_f1 = np.mean([m["f1"] for m in metrics_list])
    avg_recall = np.mean([m["recall"] for m in metrics_list])

    print(
        f"{dataset_key:<12} "
        f"{len(d['y_train']):>10} "
        f"{avg_tss:>10.4f} "
        f"{avg_f1:>10.4f} "
        f"{avg_recall:>12.4f}"
    )

    if avg_tss > best_avg_tss:
        best_avg_tss = avg_tss
        best_dataset = dataset_key


print("\nBest PatchTST cleaned Hybrid dataset:")
print(f"  Dataset : {best_dataset}")
print(f"  Avg TSS : {best_avg_tss:.4f}")
print(f"  Params  : {patchtst_label(best_params)}")

print("\n✅ All PatchTST cleaned Hybrid experiments complete.")
print(f"Results saved to: {os.path.abspath(RESULTS_DIR)}/")
print(f"Files: {sorted([f for f in os.listdir(RESULTS_DIR) if f.endswith('.txt')])}")

══════════════════════════════════════════════════════════════════════
  Classifier : Hugging Face PatchTST
  Experiment : Cleaned Hybrid data
  Best params: patch=12  stride=12  d=32  heads=4  layers=1  drop=0.2  lr=0.001  bs=128
══════════════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────────────
Dataset   : tomek
Timesteps : 288
Train     : (12428, 288, 10) | Non-SEP=12310 | SEP=118
Val       : (1780, 288, 10) | Non-SEP=1763 | SEP=17
Test      : (3559, 288, 10) | Non-SEP=3525 | SEP=34
Ratio     : 104.3:1
Params    : patch=12  stride=12  d=32  heads=4  layers=1  drop=0.2  lr=0.001  bs=128
──────────────────────────────────────────────────────────────────────
Run 1 …
Run 1: TSS=0.5757 | F1=0.0928 | Recall=0.7059 | Train=32.0s
Run 2 …
Run 2: TSS=0.4958 | F1=0.1053 | Recall=0.5882 | Train=17.5s

───────────────────────────────────────────────────────
  PatchTST | Hybrid | tomek
────────────────────────────────────

### InceptionTime

In [24]:
import os
import time
import copy
import warnings
import numpy as np

import torch
import torch.nn as nn

warnings.filterwarnings("ignore")

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")


# ══════════════════════════════════════════════════════════════
# INCEPTIONTIME MODEL
# ══════════════════════════════════════════════════════════════

class InceptionModule(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels=32,
        kernel_sizes=(9, 19, 39),
        bottleneck_channels=32
    ):
        super(InceptionModule, self).__init__()

        if in_channels > 1:
            self.bottleneck = nn.Conv1d(
                in_channels,
                bottleneck_channels,
                kernel_size=1,
                bias=False
            )
            conv_in_channels = bottleneck_channels
        else:
            self.bottleneck = nn.Identity()
            conv_in_channels = in_channels

        self.conv_list = nn.ModuleList([
            nn.Conv1d(
                conv_in_channels,
                out_channels,
                kernel_size=k,
                padding=k // 2,
                bias=False
            )
            for k in kernel_sizes
        ])

        self.maxpool_conv = nn.Sequential(
            nn.MaxPool1d(kernel_size=3, stride=1, padding=1),
            nn.Conv1d(in_channels, out_channels, kernel_size=1, bias=False)
        )

        total_out_channels = out_channels * (len(kernel_sizes) + 1)

        self.bn = nn.BatchNorm1d(total_out_channels)
        self.relu = nn.ReLU()

    def forward(self, x):
        x_bottleneck = self.bottleneck(x)

        conv_outputs = [
            conv(x_bottleneck)
            for conv in self.conv_list
        ]

        pool_output = self.maxpool_conv(x)

        out = torch.cat(conv_outputs + [pool_output], dim=1)
        out = self.bn(out)
        out = self.relu(out)

        return out


class InceptionBlock(nn.Module):
    def __init__(self, in_channels, out_channels=32, depth=3, use_residual=True):
        super(InceptionBlock, self).__init__()

        self.use_residual = use_residual
        self.depth = depth

        modules = []
        current_channels = in_channels

        for _ in range(depth):
            module = InceptionModule(
                in_channels=current_channels,
                out_channels=out_channels
            )
            modules.append(module)

            current_channels = out_channels * 4

        self.inception_modules = nn.ModuleList(modules)

        if use_residual:
            self.residual = nn.Sequential(
                nn.Conv1d(in_channels, current_channels, kernel_size=1, bias=False),
                nn.BatchNorm1d(current_channels)
            )
            self.relu = nn.ReLU()

    def forward(self, x):
        residual = x

        out = x
        for module in self.inception_modules:
            out = module(out)

        if self.use_residual:
            residual = self.residual(residual)
            out = self.relu(out + residual)

        return out


class InceptionTimeClassifier(nn.Module):
    def __init__(self, input_channels, out_channels=32, depth=6, dropout=0.3):
        super(InceptionTimeClassifier, self).__init__()

        block_depth = max(1, depth // 2)

        self.block1 = InceptionBlock(
            in_channels=input_channels,
            out_channels=out_channels,
            depth=block_depth,
            use_residual=True
        )

        block1_channels = out_channels * 4

        self.block2 = InceptionBlock(
            in_channels=block1_channels,
            out_channels=out_channels,
            depth=block_depth,
            use_residual=True
        )

        final_channels = out_channels * 4

        self.gap = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(final_channels, 1)

    def forward(self, x):
        # Input shape: (batch, timesteps, features)
        # Conv1d expects: (batch, features, timesteps)
        x = x.permute(0, 2, 1)

        x = self.block1(x)
        x = self.block2(x)

        x = self.gap(x).squeeze(-1)
        x = self.dropout(x)

        return self.fc(x).squeeze()


# ══════════════════════════════════════════════════════════════
# TRAIN AND EVALUATE INCEPTIONTIME
# ══════════════════════════════════════════════════════════════

def train_and_evaluate_inception(
    params,
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
    class_ratio,
    max_epochs=30,
    patience=3
):
    X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train, dtype=torch.float32).to(device)

    X_va = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_va = torch.tensor(y_val, dtype=torch.float32).to(device)

    X_te = torch.tensor(X_test, dtype=torch.float32).to(device)

    input_channels = X_train.shape[2]

    model = InceptionTimeClassifier(
        input_channels=input_channels,
        out_channels=params["out_channels"],
        depth=params["depth"],
        dropout=params["dropout"]
    ).to(device)

    pos_weight = torch.tensor([float(class_ratio)], dtype=torch.float32).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=params["lr"],
        weight_decay=1e-4
    )

    batch_size = params["batch_size"]
    n_samples = X_tr.shape[0]

    best_val_loss = float("inf")
    best_state = None
    patience_count = 0

    t0 = time.time()

    for epoch in range(max_epochs):
        model.train()

        perm = torch.randperm(n_samples, device=device)

        for i in range(0, n_samples, batch_size):
            idx = perm[i:i + batch_size]

            xb = X_tr[idx]
            yb = y_tr[idx]

            optimizer.zero_grad()

            logits = model(xb)
            loss = criterion(logits, yb)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(X_va)
            val_loss = criterion(val_logits, y_va).item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_count = 0
        else:
            patience_count += 1

            if patience_count >= patience:
                break

    train_time = time.time() - t0

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    t0 = time.time()

    with torch.no_grad():
        y_prob = torch.sigmoid(model(X_te)).cpu().numpy()

    y_pred = (y_prob >= 0.5).astype(int)
    infer_time = time.time() - t0

    metrics = compute_metrics(y_test.astype(int), y_pred.astype(int))
    metrics["train_time"] = train_time
    metrics["infer_time"] = infer_time

    return metrics, model

Using device: mps


In [28]:
# ══════════════════════════════════════════════════════════════
# INCEPTIONTIME FINAL EXPERIMENTS — CLEANED HYBRID 6H DATA
# Uses fixed best hyperparameters selected previously
# Best params: channels=16, depth=6, dropout=0.3, lr=0.001, batch_size=128
# Datasets: tomek, enn, nearmiss
# ══════════════════════════════════════════════════════════════

import os
import copy
import numpy as np

RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ---------------------------------------------------------
# Fixed best InceptionTime hyperparameters
# ---------------------------------------------------------
best_params = {
    "out_channels": 16,
    "depth": 6,
    "dropout": 0.3,
    "lr": 0.005,
    "batch_size": 128
}


def inception_label(p):
    return (
        f"channels={p['out_channels']}  "
        f"depth={p['depth']}  "
        f"drop={p['dropout']}  "
        f"lr={p['lr']}  "
        f"bs={p['batch_size']}"
    )


print(f"{'═' * 70}")
print("  Classifier : InceptionTime")
print("  Experiment : Cleaned Hybrid data")
print(f"  Best params: {inception_label(best_params)}")
print(f"{'═' * 70}")


# ---------------------------------------------------------
# Run final InceptionTime experiments on cleaned datasets
# ---------------------------------------------------------
all_inception_cleaned_results = {}

for dataset_key, d in cleaned_datasets.items():

    timesteps = d["X_train"].shape[1]

    train_counts = np.bincount(d["y_train"].astype(int), minlength=2)
    val_counts   = np.bincount(d["y_val"].astype(int), minlength=2)
    test_counts  = np.bincount(d["y_test"].astype(int), minlength=2)

    cr = train_counts[0] / train_counts[1]

    print("\n" + "─" * 70)
    print(f"Dataset   : {dataset_key}")
    print(f"Timesteps : {timesteps}")
    print(f"Train     : {d['X_train'].shape} | Non-SEP={train_counts[0]} | SEP={train_counts[1]}")
    print(f"Val       : {d['X_val'].shape} | Non-SEP={val_counts[0]} | SEP={val_counts[1]}")
    print(f"Test      : {d['X_test'].shape} | Non-SEP={test_counts[0]} | SEP={test_counts[1]}")
    print(f"Ratio     : {cr:.1f}:1")
    print(f"Params    : {inception_label(best_params)}")
    print("─" * 70)

    metrics_list = []

    for run in range(2):

        print(f"Run {run + 1} …", flush=True)

        metrics, _ = train_and_evaluate_inception(
            copy.deepcopy(best_params),
            d["X_train"], d["y_train"],
            d["X_val"],   d["y_val"],
            d["X_test"],  d["y_test"],
            class_ratio=cr,
            max_epochs=30,
            patience=3
        )

        metrics_list.append(metrics)

        precision_text = (
            f" | Precision={metrics['precision']:.4f}"
            if "precision" in metrics else ""
        )

        print(
            f"Run {run + 1}: "
            f"TSS={metrics['tss']:.4f} | "
            f"F1={metrics['f1']:.4f} | "
            f"Recall={metrics['recall']:.4f}"
            f"{precision_text} | "
            f"Train={metrics['train_time']:.1f}s"
        )

    all_inception_cleaned_results[dataset_key] = metrics_list

    filepath = os.path.join(RESULTS_DIR, f"inceptiontime_hybrid_{dataset_key}.txt")
    save_results(metrics_list, filepath)
    print_results(metrics_list, f"InceptionTime | Hybrid | {dataset_key}")


# ---------------------------------------------------------
# Select best cleaned dataset by average TSS
# ---------------------------------------------------------
best_dataset = None
best_avg_tss = -np.inf

print("\n" + "═" * 70)
print("InceptionTime cleaned Hybrid summary")
print("═" * 70)

print(
    f"{'Dataset':<12} "
    f"{'Train N':>10} "
    f"{'Avg TSS':>10} "
    f"{'Avg F1':>10} "
    f"{'Avg Recall':>12}"
)

print("─" * 65)

for dataset_key, metrics_list in all_inception_cleaned_results.items():

    d = cleaned_datasets[dataset_key]

    avg_tss = np.mean([m["tss"] for m in metrics_list])
    avg_f1 = np.mean([m["f1"] for m in metrics_list])
    avg_recall = np.mean([m["recall"] for m in metrics_list])

    print(
        f"{dataset_key:<12} "
        f"{len(d['y_train']):>10} "
        f"{avg_tss:>10.4f} "
        f"{avg_f1:>10.4f} "
        f"{avg_recall:>12.4f}"
    )

    if avg_tss > best_avg_tss:
        best_avg_tss = avg_tss
        best_dataset = dataset_key


print("\nBest InceptionTime cleaned Hybrid dataset:")
print(f"  Dataset : {best_dataset}")
print(f"  Avg TSS : {best_avg_tss:.4f}")
print(f"  Params  : {inception_label(best_params)}")

print("\n✅ All InceptionTime cleaned Hybrid experiments complete.")
print(f"Results saved to: {os.path.abspath(RESULTS_DIR)}/")
print(f"Files: {sorted([f for f in os.listdir(RESULTS_DIR) if f.endswith('.txt')])}")

══════════════════════════════════════════════════════════════════════
  Classifier : InceptionTime
  Experiment : Cleaned Hybrid data
  Best params: channels=16  depth=6  drop=0.3  lr=0.005  bs=128
══════════════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────────────
Dataset   : tomek
Timesteps : 288
Train     : (12428, 288, 10) | Non-SEP=12310 | SEP=118
Val       : (1780, 288, 10) | Non-SEP=1763 | SEP=17
Test      : (3559, 288, 10) | Non-SEP=3525 | SEP=34
Ratio     : 104.3:1
Params    : channels=16  depth=6  drop=0.3  lr=0.005  bs=128
──────────────────────────────────────────────────────────────────────
Run 1 …
Run 1: TSS=0.4145 | F1=0.0545 | Recall=0.6176 | Train=179.4s
Run 2 …
Run 2: TSS=0.3897 | F1=0.0955 | Recall=0.4706 | Train=151.7s

───────────────────────────────────────────────────────
  InceptionTime | Hybrid | tomek
───────────────────────────────────────────────────────
  Run 1: TP=21  TN=2809  FP=7

### Saving the ENN data

In [27]:
import os
import pickle
import numpy as np

# Save only ENN-cleaned dataset
output_dir = "./final_split_data_HybridNorm_ENN"
os.makedirs(output_dir, exist_ok=True)

d = cleaned_datasets["enn"]

def save_split(X, y, path):
    with open(path, "wb") as f:
        pickle.dump(
            {
                "X": X.astype(np.float32),
                "y": y
            },
            f
        )
    print(f"Saved: {path}")

save_split(d["X_train"], d["y_train"], os.path.join(output_dir, "train_set.pkl"))
save_split(d["X_val"],   d["y_val"],   os.path.join(output_dir, "val_set.pkl"))
save_split(d["X_test"],  d["y_test"],  os.path.join(output_dir, "test_set.pkl"))

print(f"\n✅ ENN-cleaned dataset saved to: {os.path.abspath(output_dir)}/")

Saved: ./final_split_data_HybridNorm_ENN/train_set.pkl
Saved: ./final_split_data_HybridNorm_ENN/val_set.pkl
Saved: ./final_split_data_HybridNorm_ENN/test_set.pkl

✅ ENN-cleaned dataset saved to: /Users/samskanderi/SEP_DataAugmentation/final_split_data_HybridNorm_ENN/
